In [1]:
import pandas as pd
import numpy as np
from pandas.api.types import CategoricalDtype
from collections import defaultdict

In [3]:
def preprocess(df):
    # 1. '회 이상' 문자열 제거 후 숫자형으로 변환
    count_cols2 = [
        '인입횟수_ARS_R6M',
        '이용메뉴건수_ARS_R6M',
        '방문횟수_PC_R6M',
        '방문일수_PC_R6M',
        '방문횟수_앱_R6M'
    ]

    for col in count_cols2:
        df[col] = df[col].astype(str).str.replace('회 이상', '', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(int)

    # 2. 라벨 인코딩
    le_cols = ['OS구분코드']

    for col in le_cols:
        codes, uniques = pd.factorize(df[col], sort=True)
        df[col] = codes

    return df

df1 = preprocess(pd.read_parquet('train/6.채널정보/201807_train_채널정보.parquet'))
df2 = preprocess(pd.read_parquet('train/6.채널정보/201808_train_채널정보.parquet'))
df3 = preprocess(pd.read_parquet('train/6.채널정보/201809_train_채널정보.parquet'))
df4 = preprocess(pd.read_parquet('train/6.채널정보/201810_train_채널정보.parquet'))
df5 = preprocess(pd.read_parquet('train/6.채널정보/201811_train_채널정보.parquet'))
df6 = preprocess(pd.read_parquet('train/6.채널정보/201812_train_채널정보.parquet'))

In [4]:
dfs = [df.drop(columns=['기준년월'], errors='ignore') for df in [df1, df2, df3, df4, df5, df6]]

def merge_two_avg(df_left, df_right):
    merge_keys = ['ID']
    if 'Segment' in df_left.columns and 'Segment' in df_right.columns:
        merge_keys.append('Segment')

    merged = pd.merge(df_left, df_right, on=merge_keys, how='outer', suffixes=('_left', '_right'))
    result = merged[merge_keys].copy()
    
    # 평균 계산
    for col in set(df_left.columns).union(df_right.columns):
        if col in merge_keys:
            continue
        col_left = f"{col}_left" if f"{col}_left" in merged.columns else None
        col_right = f"{col}_right" if f"{col}_right" in merged.columns else None
        
        cols_to_avg = [c for c in [col_left, col_right] if c is not None]
        result[col] = merged[cols_to_avg].mean(axis=1, skipna=True)
    
    return result

from functools import reduce
merged_df = reduce(merge_two_avg, dfs)
merged_df

C:\Users\AHN\AppData\Local\Temp\ipykernel_18684\3324607319.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[col] = merged[cols_to_avg].mean(axis=1, skipna=True)
C:\Users\AHN\AppData\Local\Temp\ipykernel_18684\3324607319.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[col] = merged[cols_to_avg].mean(axis=1, skipna=True)
C:\Users\AHN\AppData\Local\Temp\ipykernel_18684\3324607319.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which 

,ID,방문월수_모바일웹_R6M,IB문의건수_한도_B0M,IB문의건수_비밀번호_R6M,IB문의건수_카드발급_R6M,방문횟수_앱_B0M,상담건수_B0M,인입불만월수_IB_R6M,이용메뉴건수_IB_R6M,방문후경과월_모바일웹_R6M,...,IB상담건수_금감원_B0M,IB문의건수_비밀번호_B0M,당사PAY_방문월수_R6M,IB상담건수_VOC민원_R6M,방문월수_PC_R6M,인입횟수_ARS_B0M,인입후경과월_ARS,IB상담건수_VOC불만_B0M,방문일수_PC_R6M,인입일수_IB_R6M
0,TRAIN_000000,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.500,6.0,...,0.0,0.0,0.0,0.0,0.0,2.0000,0.00000,0.0,1.0,0.96875
1,TRAIN_000001,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0000,0.00000,0.0,1.0,0.00000
2,TRAIN_000002,0.0,0.0,0.0,0.0,8.0,0.0625,0.0,1.375,6.0,...,0.0,0.0,0.0,0.0,3.5,0.0625,4.03125,0.0,15.0,1.25000
3,TRAIN_000003,0.0,0.0,0.0,0.0,0.0,0.0625,0.0,2.000,6.0,...,0.0,0.0,0.0,0.0,0.0,2.0625,0.00000,0.0,1.0,2.50000
4,TRAIN_000004,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0000,0.00000,0.0,1.0,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0000,0.00000,0.0,1.0,0.00000
399996,TRAIN_399996,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.500,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0000,1.03125,0.0,1.0,0.46875
399997,TRAIN_399997,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0000,0.00000,0.0,1.0,0.00000
399998,TRAIN_399998,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0000,0.00000,0.0,1.0,0.00000


In [5]:
df_segment = pd.read_parquet('train/1.회원정보/201807_train_회원정보.parquet')[['ID', 'Segment']]

# 2. 중복 제거 (ID별 Segment가 유일하다는 전제)
df_segment = df_segment.drop_duplicates(subset='ID')

# 3. merged_df에 Segment 열 붙이기 (ID 기준)
merged_df = pd.merge(merged_df, df_segment, on='ID', how='left')
merged_df

,ID,방문월수_모바일웹_R6M,IB문의건수_한도_B0M,IB문의건수_비밀번호_R6M,IB문의건수_카드발급_R6M,방문횟수_앱_B0M,상담건수_B0M,인입불만월수_IB_R6M,이용메뉴건수_IB_R6M,방문후경과월_모바일웹_R6M,...,IB문의건수_비밀번호_B0M,당사PAY_방문월수_R6M,IB상담건수_VOC민원_R6M,방문월수_PC_R6M,인입횟수_ARS_B0M,인입후경과월_ARS,IB상담건수_VOC불만_B0M,방문일수_PC_R6M,인입일수_IB_R6M,Segment
0,TRAIN_000000,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.500,6.0,...,0.0,0.0,0.0,0.0,2.0000,0.00000,0.0,1.0,0.96875,D
1,TRAIN_000001,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000,6.0,...,0.0,0.0,0.0,0.0,0.0000,0.00000,0.0,1.0,0.00000,E
2,TRAIN_000002,0.0,0.0,0.0,0.0,8.0,0.0625,0.0,1.375,6.0,...,0.0,0.0,0.0,3.5,0.0625,4.03125,0.0,15.0,1.25000,C
3,TRAIN_000003,0.0,0.0,0.0,0.0,0.0,0.0625,0.0,2.000,6.0,...,0.0,0.0,0.0,0.0,2.0625,0.00000,0.0,1.0,2.50000,D
4,TRAIN_000004,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000,6.0,...,0.0,0.0,0.0,0.0,0.0000,0.00000,0.0,1.0,0.00000,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000,6.0,...,0.0,0.0,0.0,0.0,0.0000,0.00000,0.0,1.0,0.00000,E
399996,TRAIN_399996,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.500,6.0,...,0.0,0.0,0.0,0.0,0.0000,1.03125,0.0,1.0,0.46875,D
399997,TRAIN_399997,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000,6.0,...,0.0,0.0,0.0,0.0,0.0000,0.00000,0.0,1.0,0.00000,C
399998,TRAIN_399998,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.000,6.0,...,0.0,0.0,0.0,0.0,0.0000,0.00000,0.0,1.0,0.00000,E


In [6]:
nan_columns = merged_df.columns[merged_df.isnull().any()].tolist()

print("NaN이 포함된 열 목록:")
print(nan_columns)

NaN이 포함된 열 목록:
[]


In [7]:
ex1 = merged_df

In [8]:
missing_mask = ex1.isna() | (ex1 == -1)
missing_ratio = missing_mask.mean()

high_na = missing_ratio[missing_ratio > 0.2].index.tolist()

high_const_cols = []
threshold_const = 0.8

for col in ex1.columns:
    top_ratio = ex1[col].value_counts(normalize=True, dropna=False).values[0]
    if top_ratio > threshold_const:
        high_const_cols.append(col)

to_drop = list(set(high_na + high_const_cols))

if 'Segment' in to_drop:
    to_drop.remove('Segment')

print("삭제 대상 컬럼 (결측>20% 또는 동일값>80%):", to_drop)

ex1.drop(columns=to_drop, inplace=True)

삭제 대상 컬럼 (결측>20% 또는 동일값>80%): ['IB문의건수_한도_B0M', '인입불만월수_IB_R6M', '인입횟수_금융_IB_R6M', 'IB문의건수_선결제_B0M', 'IB문의건수_사용승인내역_R6M', '인입횟수_ARS_R6M', 'IB문의건수_BL_R6M', '방문횟수_PC_B0M', 'IB문의건수_정보변경_B0M', '방문일수_PC_B0M', 'IB문의건수_BL_B0M', '인입불만횟수_IB_R6M', '당사멤버쉽_방문월수_R6M', 'IB문의건수_CL_RV_B0M', '방문횟수_PC_R6M', '방문횟수_모바일웹_R6M', '이용메뉴건수_ARS_B0M', '당사멤버쉽_방문횟수_B0M', '방문후경과월_PC_R6M', 'IB상담건수_VOC불만_R6M', 'IB문의건수_결제_R6M', 'IB문의건수_CA_R6M', '당사PAY_방문횟수_R6M', 'IB문의건수_한도_R6M', '불만제기건수_R12M', '홈페이지_금융건수_R6M', '인입불만후경과월_IB_R6M', '방문일수_모바일웹_B0M', 'IB문의건수_명세서_B0M', 'IB상담건수_VOC_B0M', 'IB문의건수_할부_R6M', 'IB상담건수_금감원_R6M', '인입일수_ARS_B0M', '당사PAY_방문월수_R6M', 'IB상담건수_VOC민원_R6M', '인입횟수_ARS_B0M', 'IB상담건수_VOC불만_B0M', '방문일수_PC_R6M', '방문월수_모바일웹_R6M', 'IB문의건수_비밀번호_R6M', 'IB문의건수_카드발급_R6M', '상담건수_B0M', '방문후경과월_모바일웹_R6M', 'IB문의건수_CL_RV_R6M', 'IB문의건수_분실도난_B0M', 'IB문의건수_APP_B0M', 'IB문의건수_카드발급_B0M', 'IB문의건수_선결제_R6M', 'IB문의건수_할부_B0M', 'IB문의건수_포인트_B0M', 'IB문의건수_SMS_R6M', 'IB상담건수_VOC민원_B0M', 'IB문의건수_부대서비스_R6M', '불만제기건수_B0M', '인입불만일수_IB_R6M', '인

In [9]:
num_df = ex1.select_dtypes(include=[np.number]).dropna()

corr = num_df.corr().abs()

high_corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
)
high_corr_pairs.columns = ['Feature_1', 'Feature_2', 'Correlation']
high_corr_pairs = high_corr_pairs[high_corr_pairs['Correlation'] > 0.8]

if not np.issubdtype(ex1['Segment'].dtype, np.number):
    segment_map = {label: idx for idx, label in enumerate(sorted(ex1['Segment'].unique()))}
    ex1['Segment_encoded'] = ex1['Segment'].map(segment_map)
else:
    ex1['Segment_encoded'] = ex1['Segment']

segment_corr = ex1[num_df.columns].corrwith(ex1['Segment_encoded']).abs()

high_corr_pairs['Corr_with_Segment_1'] = high_corr_pairs['Feature_1'].map(segment_corr)
high_corr_pairs['Corr_with_Segment_2'] = high_corr_pairs['Feature_2'].map(segment_corr)

high_corr_pairs = high_corr_pairs.sort_values(by='Correlation', ascending=False).reset_index(drop=True)

print(f"▶ 상관계수 0.7 초과 변수쌍 수: {len(high_corr_pairs)}")
display(high_corr_pairs)


▶ 상관계수 0.7 초과 변수쌍 수: 33


,Feature_1,Feature_2,Correlation,Corr_with_Segment_1,Corr_with_Segment_2
0,방문횟수_앱_B0M,방문일수_앱_B0M,0.996212,0.161102,0.158874
1,방문횟수_앱_B0M,방문일수_앱_R6M,0.986982,0.161102,0.154988
2,인입횟수_IB_R6M,인입일수_IB_R6M,0.985868,0.134466,0.140493
3,방문일수_앱_B0M,방문일수_앱_R6M,0.985413,0.158874,0.154988
4,이용메뉴건수_IB_R6M,인입일수_IB_R6M,0.984078,0.138415,0.140493
5,방문월수_앱_R6M,방문후경과월_앱_R6M,0.968669,0.156748,0.161083
6,이용메뉴건수_IB_R6M,인입횟수_IB_R6M,0.966933,0.138415,0.134466
7,상담건수_R6M,인입일수_IB_R6M,0.965648,0.140401,0.140493
8,인입횟수_IB_R6M,상담건수_R6M,0.961109,0.134466,0.140401
9,인입월수_ARS_R6M,인입일수_ARS_R6M,0.959058,0.148659,0.149535


In [10]:
to_drop = []

for _, row in high_corr_pairs.iterrows():
    f1, f2 = row['Feature_1'], row['Feature_2']
    c1, c2 = row['Corr_with_Segment_1'], row['Corr_with_Segment_2']
    
    if pd.isna(c1) or pd.isna(c2):
        continue
    
    if c1 < c2:
        to_drop.append(f1)
    else:
        to_drop.append(f2)

to_drop = list(set(to_drop))

# 결과 출력
print(f"▶ 제거 대상 피처 수: {len(to_drop)}")
print("제거할 피처 목록:")
print(to_drop)

▶ 제거 대상 피처 수: 11
제거할 피처 목록:
['인입월수_IB_R6M', '인입월수_ARS_R6M', '상담건수_R6M', '방문일수_앱_B0M', '인입후경과월_IB_R6M', '방문월수_앱_R6M', '이용메뉴건수_IB_R6M', '방문후경과월_앱_R6M', '방문일수_앱_R6M', '인입횟수_IB_R6M', '인입일수_IB_R6M']


In [11]:
cols_to_drop = ['인입월수_IB_R6M', '인입월수_ARS_R6M', '상담건수_R6M', '방문일수_앱_B0M', '인입후경과월_IB_R6M', '방문월수_앱_R6M', '이용메뉴건수_IB_R6M', '방문후경과월_앱_R6M', '방문일수_앱_R6M', '인입횟수_IB_R6M', '인입일수_IB_R6M']
ex1.drop(columns=cols_to_drop, inplace=True)

In [12]:
cols_to_drop = ['Segment_encoded']
ex1.drop(columns=cols_to_drop, inplace=True)
cols = ex1.columns.tolist()
cols

['ID', '방문횟수_앱_B0M', '인입일수_ARS_R6M', '불만제기후경과월_R12M', '인입후경과월_ARS', 'Segment']

In [13]:
ex1.to_parquet('채널_전처리_Segment.parquet', index=False)

In [14]:
ddf1 = preprocess(pd.read_parquet('train/6.채널정보/201807_train_채널정보.parquet'))
ddf2 = preprocess(pd.read_parquet('train/6.채널정보/201808_train_채널정보.parquet'))
ddf3 = preprocess(pd.read_parquet('train/6.채널정보/201809_train_채널정보.parquet'))
ddf4 = preprocess(pd.read_parquet('train/6.채널정보/201810_train_채널정보.parquet'))
ddf5 = preprocess(pd.read_parquet('train/6.채널정보/201811_train_채널정보.parquet'))
ddf6 = preprocess(pd.read_parquet('train/6.채널정보/201812_train_채널정보.parquet'))

In [15]:

dfs = [ddf1, ddf2, ddf3, ddf4, ddf5, ddf6]
for i in range(len(dfs)):
    if '기준년월' in dfs[i].columns:
        dfs[i] = dfs[i].drop(columns=['기준년월'])

# ID 기준으로 병합 후 평균
from functools import reduce

merged_df = reduce(
    lambda left, right: pd.merge(left, right, on='ID', how='outer', suffixes=('', '_dup')),
    dfs
)

# 같은 이름의 열 평균 구하기
from collections import defaultdict
import pandas as pd

result = pd.DataFrame()
result['ID'] = merged_df['ID']

# 열 이름 모아 평균 구하기
col_dict = defaultdict(list)
for col in merged_df.columns:
    if col != 'ID':
        base_col = col.split('_dup')[0]
        col_dict[base_col].append(col)

for base_col, cols in col_dict.items():
    result[base_col] = merged_df[cols].mean(axis=1, skipna=True)

# 결과 확인
print(result.head())

C:\Users\AHN\AppData\Local\Temp\ipykernel_18684\3643205472.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[base_col] = merged_df[cols].mean(axis=1, skipna=True)
C:\Users\AHN\AppData\Local\Temp\ipykernel_18684\3643205472.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[base_col] = merged_df[cols].mean(axis=1, skipna=True)
C:\Users\AHN\AppData\Local\Temp\ipykernel_18684\3643205472.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whic

             ID  인입횟수_ARS_R6M  이용메뉴건수_ARS_R6M  인입일수_ARS_R6M  인입월수_ARS_R6M  \
0  TRAIN_000000          10.0       13.846154      9.730769           6.0   
1  TRAIN_000001           1.0        1.000000      0.000000           0.0   
2  TRAIN_000002           1.0        1.000000      1.192308           1.0   
3  TRAIN_000003          10.0       13.846154     10.384615           6.0   
4  TRAIN_000004           1.0        1.000000      0.000000           0.0   

   인입후경과월_ARS  인입횟수_ARS_B0M  이용메뉴건수_ARS_B0M  인입일수_ARS_B0M  방문횟수_PC_R6M  ...  \
0    0.000000      2.000000        6.000000      2.000000     1.000000  ...   
1    0.000000      0.000000        0.000000      0.000000     1.000000  ...   
2    2.884615      0.076923        0.192308      0.038462    11.923077  ...   
3    0.000000      2.192308        6.000000      2.000000     1.000000  ...   
4    0.000000      0.000000        0.000000      0.000000     1.000000  ...   

   당사PAY_방문횟수_R6M  당사PAY_방문월수_R6M  당사멤버쉽_방문횟수_B0M  당사멤버쉽_방문횟수_

C:\Users\AHN\AppData\Local\Temp\ipykernel_18684\3643205472.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[base_col] = merged_df[cols].mean(axis=1, skipna=True)


In [16]:
cols = ['ID', '방문횟수_앱_B0M', '인입일수_ARS_R6M', '불만제기후경과월_R12M', '인입후경과월_ARS']

result = result[cols]
result

,ID,방문횟수_앱_B0M,인입일수_ARS_R6M,불만제기후경과월_R12M,인입후경과월_ARS
0,TRAIN_000000,0.000000,9.730769,12.0,0.000000
1,TRAIN_000001,0.000000,0.000000,12.0,0.000000
2,TRAIN_000002,7.769231,1.192308,12.0,2.884615
3,TRAIN_000003,0.000000,10.384615,12.0,0.000000
4,TRAIN_000004,0.000000,0.000000,0.0,0.000000
...,...,...,...,...,...
399995,TRAIN_399995,0.000000,0.000000,0.0,0.000000
399996,TRAIN_399996,0.000000,0.615385,12.0,2.384615
399997,TRAIN_399997,0.000000,0.000000,12.0,0.000000
399998,TRAIN_399998,0.000000,0.000000,0.0,0.000000


In [19]:
result.to_parquet('채널_전처리_test.parquet', index=False)

In [20]:
nan_columns = result.columns[result.isnull().any()].tolist()

print("NaN이 포함된 열 목록:")
print(nan_columns)

NaN이 포함된 열 목록:
[]
